# 🌎 PAHO (Pan American Health Organization) Data Access

This notebook demonstrates how to access health data from the **Pan American Health Organization (PAHO)**, which covers all countries in the Americas (North America, Central America, South America, and the Caribbean).

## Overview

PAHO is the World Health Organization's Regional Office for the Americas. It provides:
- Immunization coverage data
- Disease surveillance (malaria, dengue, chikungunya, zika, etc.)
- Mortality and demographic data
- Health system indicators

**Website:** https://www.paho.org/en/data

## 📦 Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Add scripts directory to path
sys.path.insert(0, str(Path.cwd().parent.parent / 'scripts'))

from accessors import PAHOAccessor

# Initialize accessor
paho = PAHOAccessor()

print("✅ PAHO Accessor initialized successfully!")

## 🌍 Exploring PAHO Coverage

PAHO covers 35 member states across 6 subregions:

In [ ]:
# List all PAHO member countries
countries = paho.list_countries()
print(f"Total PAHO member countries: {len(countries)}")
print("\nFirst 10 countries:")
countries.head(10)

In [ ]:
# List subregions
subregions = paho.list_subregions()

print("PAHO Subregions:")
for subregion in subregions['subregion'].unique():
    n_countries = len(subregions[subregions['subregion'] == subregion])
    country_list = subregions[subregions['subregion'] == subregion]['country_code'].tolist()
    print(f"\n📍 {subregion} ({n_countries} countries):")
    print(f"   {', '.join(country_list)}")

## 💉 Immunization Coverage Data

PAHO tracks vaccination coverage across the Americas. Let's explore the available vaccines and get coverage data.

In [ ]:
# List available vaccines
vaccines = paho.list_vaccines()
print("Available vaccines for tracking:")
vaccines

In [ ]:
# Get DTP3 (Diphtheria-Tetanus-Pertussis) coverage for Southern Cone countries
# Note: This requires internet connection and WHO API access

try:
    dtp3_coverage = paho.get_immunization_coverage(
        vaccines=['DTP3'],
        subregion='Southern Cone',
        years=list(range(2015, 2024))
    )
    
    if not dtp3_coverage.empty:
        print(f"Retrieved {len(dtp3_coverage)} records")
        print("\nSample data:")
        print(dtp3_coverage.head(10))
    else:
        print("No data retrieved. This may be due to API limitations.")
        
except Exception as e:
    print(f"Note: API access may be limited: {e}")
    print("The accessor is properly configured and will work when API is available.")

In [ ]:
# Visualize immunization coverage (if data available)
if not dtp3_coverage.empty and 'coverage' in dtp3_coverage.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for country in dtp3_coverage['country_code'].unique():
        country_data = dtp3_coverage[dtp3_coverage['country_code'] == country]
        ax.plot(country_data['year'], country_data['coverage'], 
                marker='o', label=country, linewidth=2)
    
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('DTP3 Coverage (%)', fontsize=12)
    ax.set_title('DTP3 Immunization Coverage - Southern Cone Countries', fontsize=14)
    ax.legend(title='Country', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 100)
    
    plt.tight_layout()
    plt.show()

## 🦟 Malaria Surveillance Data

Malaria is a priority disease for PAHO surveillance, particularly in the Amazon region and parts of Central America.

In [ ]:
# Get malaria-endemic countries in PAHO
malaria_countries = ['BRA', 'COL', 'ECU', 'VEN', 'GUY', 'SUR', 'PER', 'BOL']

print("Malaria-endemic countries in PAHO region:")
for code in malaria_countries:
    name = paho.COUNTRIES.get(code, code)
    print(f"  - {name} ({code})")

In [ ]:
# Get malaria incidence data
try:
    malaria_data = paho.get_malaria_incidence(
        countries=malaria_countries,
        years=[2020, 2021, 2022]
    )
    
    if not malaria_data.empty:
        print(f"\nRetrieved {len(malaria_data)} malaria records")
        print("\nSample data:")
        print(malaria_data.head())
    else:
        print("No malaria data retrieved from API")
        
except Exception as e:
    print(f"Note: {e}")

## 🦠 Dengue Surveillance

Dengue is endemic throughout the PAHO region. The accessor provides a structure for accessing PAHO epidemiological bulletins.

In [ ]:
# List priority diseases for PAHO surveillance
diseases = paho.list_diseases()
print("Priority diseases for PAHO surveillance:")
diseases

In [ ]:
# Get dengue data structure
dengue_data = paho.get_dengue_data(
    countries=['BRA', 'COL', 'MEX'],
    years=[2022]
)

print("Dengue data structure:")
print(dengue_data)

print("\nNote: Actual case data requires manual extraction from PAHO epidemiological bulletins.")
print("See: https://www.paho.org/en/documents")

## 📊 Health Indicators

Access key health indicators for PAHO countries.

In [ ]:
# Get life expectancy data for a selection of countries
sample_countries = ['BRA', 'MEX', 'ARG', 'COL', 'CHL', 'CRI']

try:
    life_exp = paho.get_health_indicators(
        indicator='LIFE_EXPECTANCY',
        countries=sample_countries,
        years=[2019, 2020, 2021]
    )
    
    if not life_exp.empty:
        print("Life Expectancy Data:")
        print(life_exp.head(10))
        
        # Create comparison table
        comparison = paho.compare_countries(
            indicator='LIFE_EXPECTANCY',
            countries=sample_countries,
            years=[2019, 2020, 2021]
        )
        print("\nComparison Table:")
        print(comparison)
        
except Exception as e:
    print(f"Note: {e}")

## 🗺️ Regional Summaries

Compare health indicators across PAHO subregions.

In [ ]:
# Get regional summary for life expectancy
try:
    regional_summary = paho.get_regional_summary(
        indicator='LIFE_EXPECTANCY',
        year=2019
    )
    
    if not regional_summary.empty:
        print("Life Expectancy by PAHO Subregion (2019):")
        print(regional_summary[['subregion', 'mean_value', 'median_value', 'n_countries']])
        
        # Visualize
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(regional_summary['subregion'], regional_summary['mean_value'])
        ax.set_xlabel('Life Expectancy (years)')
        ax.set_title('Average Life Expectancy by PAHO Subregion (2019)')
        ax.grid(True, alpha=0.3, axis='x')
        plt.tight_layout()
        plt.show()
        
except Exception as e:
    print(f"Note: {e}")

## 🔧 Advanced Usage

### Combining with Other Accessors

You can combine PAHO data with other accessors for comprehensive analysis:

In [ ]:
# Example: Compare Brazil data from PAHO and DataSUS
from accessors import DataSUSAccessor, WHOAccessor

print("📊 Combining PAHO, WHO, and DataSUS data sources")
print("="*60)

# Check available accessors
from accessors import check_libraries
status = check_libraries()

for lib, info in status.items():
    symbol = "✅" if info['available'] else "❌"
    print(f"{symbol} {lib}: {info['description']}")

## 📚 References

- **PAHO Data Portal:** https://www.paho.org/en/data
- **PAHO Immunization:** https://www.paho.org/en/topics/immunization
- **WHO Global Health Observatory:** https://www.who.int/data/gho
- **PAHO Publications:** https://www.paho.org/en/documents

## 📝 Notes

1. **Data Availability:** Some PAHO data requires manual extraction from epidemiological bulletins.
2. **WHO Integration:** This accessor uses WHO APIs (GHO, Immunization) as primary data sources for PAHO region.
3. **Caching:** Consider implementing caching for repeated API calls.
4. **Rate Limits:** Be mindful of API rate limits when making multiple requests.